PREPROCESSING

In [1]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pydub import AudioSegment
from pathlib import Path
from datetime import datetime
import traceback
from scipy.signal import butter, sosfiltfilt
import soundfile as sf
import pandas as pd
import os
import matplotlib.dates as mdates
import sys
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import noisereduce as nr
import shutil
import re
from datetime import datetime, timedelta

c:\Users\chris\Desktop\Fase2Tesi\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
c:\Users\chris\Desktop\Fase2Tesi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=sys.maxsize)

NORMALIZED_AUDIO=False
normalized_str="[NORMALIZED_AUDIO]" * NORMALIZED_AUDIO
NOISE_REDUCTION=False
noise_reduction_str="[NOISE_REDUCTION]" * NOISE_REDUCTION
HIGH_PASS_FILTER=False
high_pass_filter_str="[HIGH_PASS_FILTER]" * HIGH_PASS_FILTER

INPUT_DIR = Path("AudioSamplesFiltered")
CSV_INPUT_PATH = Path("audio_samples_filtered_metadata.csv")
CSV_OUTPUT_PATH = Path(f"{normalized_str}{noise_reduction_str}{high_pass_filter_str}audio_samples_filtered_and_preprocessed_metadata.csv")
NOISE_PATH = Path("noise.wav")
OUTPUT_DIR_PREPROCESSED = Path(f"{normalized_str}{noise_reduction_str}{high_pass_filter_str}AudioSamplesPreprocessed")

SAMPLE_RATE = 16000
FRAME_SIZE = 2048
HOP_LENGTH = 512
TARGET_DB = -1.0
target_amplitude = librosa.db_to_amplitude(TARGET_DB)
CUTOFF_FREQ = 300   #100
NOISE_REDUCTION_PROPORTION = 0.8


In [3]:
if OUTPUT_DIR_PREPROCESSED.exists():
    shutil.rmtree(OUTPUT_DIR_PREPROCESSED)  # Rimuove la cartella e tutto il contenuto

OUTPUT_DIR_PREPROCESSED.mkdir(parents=True) # La ricrea vuota

In [4]:
def highpassfilter(data, cutoff, sr, order=5):
    # Nyquist frequency è la metà della frequenza di campionamento
    nyq = 0.5 * sr
    if cutoff >= nyq:
        raise ValueError("La frequenza di taglio deve essere inferiore alla frequenza di Nyquist.")
    normal_cutoff = cutoff / nyq
    
    # Crea il filtro in formato SOS (Second-Order Sections) 
    # È più stabile numericamente rispetto al formato classico
    sos = butter(order, normal_cutoff, btype='high', analog=False, output='sos')
    
    # Applica il filtro
    filtered_data = sosfiltfilt(sos, data)
    return filtered_data

In [5]:
def preprocess(audio, offset_to_cut=0.5, high_pass_filter=False, normalize=False, noise_signal=None):
    # 1. RIMOZIONE COMPONENTE DC
    #print("Rimozione componente DC...")
    #audio = audio - np.mean(audio)

    # 2. FILTRO PASSA ALTO
    print("Applicazione filtro passa alto... (con rimozione componente DC)")
    audio = highpassfilter(audio, cutoff=CUTOFF_FREQ, sr=SAMPLE_RATE, order=5)

    # 3. TAGLIO
    print(f"Taglio {offset_to_cut} secondi dall'inizio e dalla fine...")
    offset = int(offset_to_cut * SAMPLE_RATE)
    audio = audio[offset:-offset]

    if noise_signal is not None:
        print("Riduzione del rumore...")
        noise_signal = noise_signal - np.mean(noise_signal)
        if high_pass_filter:
            noise_signal = highpassfilter(noise_signal, cutoff=CUTOFF_FREQ, sr=SAMPLE_RATE, order=5)
        noise_signal =  noise_signal[offset:-offset]
        audio = nr.reduce_noise(y=audio, sr=SAMPLE_RATE, y_noise=noise_signal, n_fft=FRAME_SIZE, hop_length=HOP_LENGTH, prop_decrease=NOISE_REDUCTION_PROPORTION, stationary=False)
    
    if normalize:
        print("Normalizzazione audio...")
        audio = librosa.util.normalize(audio) * target_amplitude
    return audio

In [6]:
audio_files = list(INPUT_DIR.glob("*.wav"))

total_files = len(audio_files)

In [7]:
df_metadati = pd.read_csv(CSV_INPUT_PATH)
df_metadati['timestamp'] = pd.to_datetime(df_metadati['timestamp'], format='%Y-%m-%d %H:%M:%S%z')
new_rows = []

In [8]:
noise, _ = librosa.load(NOISE_PATH, sr=SAMPLE_RATE)

# 1. RIMOZIONE COMPONENTE DC 
#noise = noise - np.mean(noise)

In [ ]:
for i, filepath in enumerate(audio_files, 1):
    print(f'[{i}/{total_files}] Elaborando: {filepath.name}')
    
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE)

    noise = noise if NOISE_REDUCTION else None

    preprocessed_audio=preprocess(audio, offset_to_cut=0.5, high_pass_filter=HIGH_PASS_FILTER, normalize=NORMALIZED_AUDIO, noise_signal=noise)    #normalize=False e noise_signal=noise
    
    #output_path = OUTPUT_DIR_PREPROCESSED / filepath.name
    #normalized_output_path = OUTPUT_DIR_PREPROCESSED_NORMALIZED / filepath.name

    #sf.write(output_path, preprocessed_audio, SAMPLE_RATE)
    #sf.write(normalized_output_path, normalized_preprocessed_audio, SAMPLE_RATE)

    segment_duration=1  # Durata di ciascun segmento in secondi
    segment_length = SAMPLE_RATE * segment_duration  # sr campioni = segment_duration secondi
    segments = [preprocessed_audio[i:i + segment_length] for i in range(0, len(preprocessed_audio), segment_length)]
    row = df_metadati.loc[df_metadati['filename'] == filepath.name].iloc[0]
    end_time = row['timestamp']
    other_metadata = row.drop(['filename', 'timestamp']).to_dict()

    # 2. Salva ogni segmento
    for j, segment in enumerate(segments):
        if len(segment) == segment_length:
            offset = (len(segments)-j-1)*segment_duration
            current_time = end_time - timedelta(seconds=offset)
            new_ts_str = current_time.strftime('%Y-%m-%dT%H-%M-%S')
            new_filename = f"audio_{new_ts_str}.wav"
            
            # 4. Costruiamo la nuova riga finale
            nuova_riga = {
                'filename': new_filename,
                'timestamp': current_time,
                **other_metadata  # Espande tutti i campi rimanenti dal CSV originale
            }
            new_rows.append(nuova_riga)
            
            output_filename = OUTPUT_DIR_PREPROCESSED / new_filename
            print(f"Audio: MIN: {np.min(segment):.8f}, MAX: {np.max(segment):.8f}")
            sf.write(output_filename, segment, SAMPLE_RATE, subtype='FLOAT')
            print(f'Salvato: {output_filename}')
    

[1/2398] Elaborando: audio_2026-02-23T09-26-59.wav
Applicazione filtro passa alto... (con rimozione componente DC)
Taglio 0.5 secondi dall'inizio e dalla fine...
Audio: MIN: -0.00079762, MAX: 0.00072515
Salvato: AudioSamplesPreprocessed\audio_2026-02-23T09-26-56.wav
Audio: MIN: -0.00064843, MAX: 0.00067751
Salvato: AudioSamplesPreprocessed\audio_2026-02-23T09-26-57.wav
Audio: MIN: -0.00068695, MAX: 0.00076804
Salvato: AudioSamplesPreprocessed\audio_2026-02-23T09-26-58.wav
Audio: MIN: -0.00086543, MAX: 0.00083071
Salvato: AudioSamplesPreprocessed\audio_2026-02-23T09-26-59.wav
[2/2398] Elaborando: audio_2026-02-23T09-28-11.wav
Applicazione filtro passa alto... (con rimozione componente DC)
Taglio 0.5 secondi dall'inizio e dalla fine...
Audio: MIN: -0.00075328, MAX: 0.00074104
Salvato: AudioSamplesPreprocessed\audio_2026-02-23T09-28-08.wav
Audio: MIN: -0.00076459, MAX: 0.00078532
Salvato: AudioSamplesPreprocessed\audio_2026-02-23T09-28-09.wav
Audio: MIN: -0.00122305, MAX: 0.00118932
Salva

In [10]:
df_output = pd.DataFrame(new_rows)
df_output.to_csv(CSV_OUTPUT_PATH, index=False)